In [ ]:
import pandas as pd
import re
with open('_study.txt', 'r', encoding='utf-8') as f:
    data = f.read()
pattern = r'(\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}(?:\s?[APMapm]{2})?)\s-\s(.*?):\s(.*)'
matches = re.findall(pattern, data)
dates = []
users = []
messages = []
for match in matches:    
    dates.append(match[0])
    users.append(match[1])
    messages.append(match[2])
df = pd.DataFrame({
    'datetime': dates,
    'user': users,
    'message': messages
})

df['datetime'] = (
    df['datetime']
    .str.replace('\u202f', ' ', regex=False)
    .str.strip()
)

df['datetime'] = pd.to_datetime(
    df['datetime'],
    format='mixed'
)
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day

df['hour'] = df['datetime'].dt.hour
df['minute'] = df['datetime'].dt.minute

df['month_name'] = df['datetime'].dt.month_name()
df['day_name'] = df['datetime'].dt.day_name()

df['date'] = df['datetime'].dt.date
df['time'] = df['datetime'].dt.time



In [ ]:
# active users
print((df['user'].value_counts().head(5)/df.shape[0])*100)

In [ ]:
# timeline analysis
import seaborn as sns
import matplotlib.pyplot as plt

daily = df.groupby('date').count()['message']
sns.lineplot(data=daily)
plt.xticks(rotation='vertical')


In [ ]:
monthly_timeline = df.groupby(
    ['year', 'month_name']
).count()['message'].reset_index()

In [ ]:
sns.lineplot(data=monthly_timeline,x='month_name',y='message')

In [ ]:
pivot = df.pivot_table(
    index='day_name',
    columns='hour',
    values='message',
    aggfunc='count'
)
pivot

In [ ]:
days = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday'
]

pivot = pivot.reindex(days)

In [ ]:
plt.figure(figsize=(15,6))

sns.heatmap(
    pivot,
    cmap='YlGnBu'
)

plt.title("WhatsApp Activity Heatmap")

plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Combine all messages into one string
text = " ".join(df['message'])

# Create WordCloud object
wc = WordCloud(
    width=800,
    height=400,
    background_color='white',
    min_font_size=10
)

# Generate word cloud
wordcloud = wc.generate(text)

# Plot
plt.figure(figsize=(15,6))

plt.imshow(wordcloud)

plt.axis('off')

plt.show()

In [ ]:
import emoji

emojis = []

for msg in df['message']:
    
    for c in msg:
        
        if c in emoji.EMOJI_DATA:
            emojis.append(c)
from collections import Counter

Counter(emojis).most_common(10)

In [ ]:
selected_user = 'USER'

user_df = df[df['user'] == selected_user]
monthly_timeline = user_df.groupby(
    ['year','month','month_name']
).count()['message'].reset_index()

monthly_timeline['time'] = (
    monthly_timeline['month_name']
    + "-"
    + monthly_timeline['year'].astype(str)
)
import matplotlib.pyplot as plt

plt.plot(
    monthly_timeline['time'],
    monthly_timeline['message']
)

plt.xticks(rotation='vertical')

plt.show()

In [ ]:
pivot = user_df.pivot_table(
    index='day_name',
    columns='hour',
    values='message',
    aggfunc='count'
)

In [ ]:
import seaborn as sns

sns.heatmap(pivot)

plt.show()

In [ ]:
words = []

for message in user_df['message']:
    words.extend(message.split())

from collections import Counter

Counter(words).most_common(20)

In [ ]:
import re

# Detect media messages (works for both export types)
# "with media" export: filename.jpg (file attached)
# "without media" export: <Media omitted>

media_pattern = re.compile(
    r'<[Mm]edia omitted>|'
    r'\S+\.(jpg|jpeg|png|mp4|mkv|opus|mp3|aac|pdf|docx|gif|webp)\s*\(file attached\)',
    re.IGNORECASE
)

df['is_media'] = df['message'].str.contains(media_pattern)

# Media type classifier
def get_media_type(msg):
    msg = msg.lower()
    if re.search(r'\.(jpg|jpeg|png|gif|webp)', msg):
        return 'Image'
    elif re.search(r'\.(mp4|mkv|avi)', msg):
        return 'Video'
    elif re.search(r'\.(opus|mp3|aac)', msg):
        return 'Audio'
    elif re.search(r'\.(pdf|docx|txt|xlsx)', msg):
        return 'Document'
    elif '<media omitted>' in msg:
        return 'Unknown (omitted)'
    return None

df['media_type'] = df['message'].apply(
    lambda m: get_media_type(m) if media_pattern.search(m) else None
)

print("Total media messages:", df['is_media'].sum())
print("\nMedia type breakdown:")
print(df['media_type'].value_counts())

In [ ]:
media_df = df[df['is_media'] == True]

media_per_user = media_df['user'].value_counts().reset_index()
media_per_user.columns = ['User', 'Media Count']

import matplotlib.pyplot as plt
import seaborn as sns

# plt.figure(figsize=(12, 5))
# sns.barplot(data=media_per_user.head(10), x='Media Count', y='User', palette='magma')
# plt.title('Top 10 Users by Media Shared')
# plt.tight_layout()
# plt.show()

In [ ]:
type_counts = df['media_type'].value_counts().dropna()

plt.figure(figsize=(7, 7))
plt.pie(
    type_counts,
    labels=type_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=['#667eea', '#f093fb', '#4facfe', '#43e97b', '#fa709a']
)
plt.title('Media Type Distribution')
plt.show()

In [ ]:
media_daily = media_df.groupby('date')['message'].count().reset_index()
media_daily.columns = ['Date', 'Media Messages']

plt.figure(figsize=(15, 4))
plt.plot(media_daily['Date'], media_daily['Media Messages'], color='#f093fb', linewidth=1.5)
plt.fill_between(media_daily['Date'], media_daily['Media Messages'], alpha=0.2, color='#f093fb')
plt.title('Daily Media Sharing Timeline')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
media_pivot = media_df.pivot_table(
    index='day_name',
    columns='hour',
    values='message',
    aggfunc='count'
).reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])

plt.figure(figsize=(15, 5))
sns.heatmap(media_pivot, cmap='RdPu', linewidths=0.3)
plt.title('Media Sharing Heatmap (Day × Hour)')
plt.show()

In [ ]:
selected_user = 'USER'  # change as needed

user_media = df[(df['user'] == selected_user) & (df['is_media'] == True)]

print(f"=== Media Stats for {selected_user} ===")
print(f"Total media sent : {len(user_media)}")
print(f"% of their msgs  : {len(user_media)/len(df[df['user']==selected_user])*100:.1f}%")
print("\nBreakdown by type:")
print(user_media['media_type'].value_counts())